# 01 — Preprocessing

Data upload, cleaning, the train/test leakage audit and fix, and feature engineering. Real code from `backend/app/ml/cleaning.py`, `preprocessing.py`, `feature_engineering.py`. Part of the split requested for the project defense (Notebook / Preprocessing / Models / Training / Evaluation / Inference / Backend / Frontend / Utility files).

In [ ]:
import os
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from wherever this notebook actually lives to find the real
    project root (the folder containing both backend/app/ and data/), so every
    relative path used below resolves correctly regardless of which folder
    this notebook is opened from."""
    for candidate in [start, *start.parents]:
        if (candidate / "backend" / "app").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(
        "Could not locate the Baseera project root (a folder containing both "
        "backend/app/ and data/) above this notebook's location."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
print("Project root:", PROJECT_ROOT)


## Raw data upload

The raw Olist dataset: 9 CSVs under `data/raw/`, loaded via `app/ml/data_loading.py::load_all_olist_tables()` -- explicit dtypes, validated primary keys.

In [ ]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("data/raw")
for f in sorted(RAW_DIR.glob("*.csv")):
    df = pd.read_csv(f, nrows=0)
    n_rows = sum(1 for _ in open(f, encoding="utf-8")) - 1
    print(f"{f.name:45s} {n_rows:>7,} rows   columns: {list(df.columns)[:4]}...")


In [ ]:
import matplotlib.pyplot as plt

counts = {}
for f in sorted(RAW_DIR.glob("*.csv")):
    label = f.name.replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")
    with open(f, encoding="utf-8") as fh:
        counts[label] = sum(1 for _ in fh) - 1

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(counts.keys(), counts.values(), color="#4c72b0")
ax.set_ylabel("Rows")
ax.set_title("Raw Olist tables — row counts (data/raw/)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## Cleaning pipeline

`app/ml/cleaning.py`, orchestrated by `backend/scripts/run_pipeline.py`: duplicate-row removal, dtype correction, geolocation compression, state-code standardization, memory-usage optimization (float64->float32, int64->smaller int where safe).

In [ ]:
import inspect
from app.ml import cleaning

print([name for name, obj in inspect.getmembers(cleaning, inspect.isfunction) if not name.startswith("_")])


## The train/test leakage bug -- found and fixed

Auditing the original notebook's dataset-splitting step (see `DATA_LEAKAGE_AUDIT.md`) found a real, quantified bug: it split first, then deduplicated into a dataframe the already-split `X_train/X_val/X_test` never got rebuilt from. Every downstream metric in the original notebook was computed on a test set containing rows the model had memorized.

**The fix** (`app/ml/preprocessing.py`): normalize text -> resolve conflicting labels -> deduplicate -> **then** split (never the other order).

In [ ]:
from app.ml.preprocessing import normalize_review_text, remove_duplicate_reviews, split_sentiment_dataset

# 1. normalize (lowercase + collapsed whitespace) catches near-duplicates exact-match dedup misses
# 2. drop rows where the SAME normalized text has both a Positive and a Negative label (unresolvable)
# 3. THEN stratified 70/10/20 split, random_state=42
# 4. save every split's review_id + text_hash to artifacts/split_manifest.json for reproducibility
print(normalize_review_text, remove_duplicate_reviews, split_sentiment_dataset)


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

corrected = json.load(open("results/data_leakage_findings.json", encoding="utf-8"))["split_overlap"]["sizes"]
# Original notebook's leaky split sizes (DATA_LEAKAGE_AUDIT.md, cell 119) -- kept here only
# for this before/after comparison; that split was never valid and was never re-derived.
leaky = {"train": 26642, "val": 3807, "test": 7613}

splits = ["train", "val", "test"]
x = np.arange(len(splits))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width / 2, [leaky[s] for s in splits], width, label="Original (leaky)", color="#c44e52")
ax.bar(x + width / 2, [corrected[s] for s in splits], width, label="Corrected (dedup'd)", color="#55a868")
ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in splits])
ax.set_ylabel("Rows")
ax.set_title("Split sizes before/after the leakage fix")
ax.legend()
plt.tight_layout()
plt.show()


## Feature engineering

`backend/scripts/run_pipeline.py::stage_clean_and_build_features()` builds 5 canonical, enriched Parquet datasets from the raw tables, each with derived columns the raw data doesn't have directly (e.g. `delivery_delay_days`, `order_count` per customer). See `app/ml/feature_engineering.py`.

In [ ]:
import pandas as pd

orders = pd.read_parquet("data/processed/orders_enriched.parquet")
reviews = pd.read_parquet("data/processed/reviews_enriched.parquet")
customers = pd.read_parquet("data/processed/customers_enriched.parquet")

print("orders_enriched:   ", orders.shape, "-- e.g. columns:", list(orders.columns)[:6])
print("reviews_enriched:  ", reviews.shape, "-- includes delivery_delay_days")
print("customers_enriched:", customers.shape, "-- e.g. order_count, total_spend, is_repeat_customer")
